# MediaPulse — Phase 2: YouTube Dataset Inspection & Profiling

**Project:** MediaPulse — Media Performance & Audience Analytics  
**Target Role:** Wipro Analyst L1 (Business Analyst / Data Analyst - Media)  
**Objective:** Conduct an exhaustive, reproducible data quality inspection of the authentic YouTube India trending video dataset (`INvideos.csv` and `IN_category_id.json`).

---

## Learning Objectives for Aspiring Media Data Analysts:
1. Understand raw media dataset structures and observation mechanisms.
2. Master core Pandas profiling methods (`read_csv`, `info()`, `describe()`, `isnull().sum()`, `duplicated()`).
3. Distinguish between technical scraping duplicates and legitimate multi-day trending snapshots.
4. Identify and diagnose real-world data corruption (e.g., Microsoft Excel `#NAME?` formula bug).
5. Parse non-standard date formats (`YY.DD.MM` vs ISO 8601 timestamps).
6. Validate external JSON taxonomies and handle unmapped category IDs.

### 1. Environment Setup and Library Imports

We begin by importing the required data manipulation libraries (`pandas`, `json`, `pathlib`).

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Configure display options for readable tabular output
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

DATA_DIR = Path('../data/raw')
CSV_PATH = DATA_DIR / 'INvideos.csv'
JSON_PATH = DATA_DIR / 'IN_category_id.json'

print(f"CSV Dataset Path: {CSV_PATH.resolve()}")
print(f"JSON Taxonomy Path: {JSON_PATH.resolve()}")
print(f"CSV Exists: {CSV_PATH.exists()}")
print(f"JSON Exists: {JSON_PATH.exists()}")

CSV Dataset Path: D:\MediaPulse\data\raw\INvideos.csv
JSON Taxonomy Path: D:\MediaPulse\data\raw\IN_category_id.json
CSV Exists: True
JSON Exists: True


### 2. Loading the Raw Datasets

We load `INvideos.csv` into a Pandas DataFrame and parse `IN_category_id.json` into a Python dictionary.

In [2]:
# Load raw CSV
df = pd.read_csv(CSV_PATH)
print(f"Raw Dataset Loaded: {df.shape[0]:,} rows and {df.shape[1]} columns.")

# Load category JSON
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    category_json = json.load(f)

# Build category mapping dictionary: {category_id: category_title}
category_map = {
    int(item['id']): item['snippet']['title']
    for item in category_json.get('items', [])
}

print(f"Loaded {len(category_map)} category definitions from taxonomy JSON.")
print("Sample Category Mappings:", dict(list(category_map.items())[:5]))

Raw Dataset Loaded: 37,352 rows and 16 columns.
Loaded 31 category definitions from taxonomy JSON.
Sample Category Mappings: {1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports'}


### 3. Schema & Column Type Inspection

Let us examine the first 5 records and inspect the native column types using `df.info()`.

In [3]:
# Display top 3 rows
df.head(3)

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,kzwfHumJyYc,17.14.11,Sharry Mann: Cute Munda ( Song Teaser) | Parmi...,Lokdhun Punjabi,1,2017-11-12T12:20:39.000Z,"sharry mann|""sharry mann new song""|""sharry man...",1096327,33966,798,882,https://i.ytimg.com/vi/kzwfHumJyYc/default.jpg,False,False,False,Presenting Sharry Mann latest Punjabi Song Cu...
1,zUZ1z7FwLc8,17.14.11,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दं...",HJ NEWS,25,2017-11-13T05:43:56.000Z,"पीरियड्स के समय|""पेट पर पति करता ऐसा""|""देखकर द...",590101,735,904,0,https://i.ytimg.com/vi/zUZ1z7FwLc8/default.jpg,True,False,False,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दं..."
2,10L1hZ9qa58,17.14.11,Stylish Star Allu Arjun @ ChaySam Wedding Rece...,TFPC,24,2017-11-12T15:48:08.000Z,Stylish Star Allu Arjun @ ChaySam Wedding Rece...,473988,2011,243,149,https://i.ytimg.com/vi/10L1hZ9qa58/default.jpg,False,False,False,Watch Stylish Star Allu Arjun @ ChaySam Weddin...


In [4]:
# Inspect column data types and memory footprint
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37352 entries, 0 to 37351
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   video_id                37352 non-null  str  
 1   trending_date           37352 non-null  str  
 2   title                   37352 non-null  str  
 3   channel_title           37352 non-null  str  
 4   category_id             37352 non-null  int64
 5   publish_time            37352 non-null  str  
 6   tags                    37352 non-null  str  
 7   views                   37352 non-null  int64
 8   likes                   37352 non-null  int64
 9   dislikes                37352 non-null  int64
 10  comment_count           37352 non-null  int64
 11  thumbnail_link          37352 non-null  str  
 12  comments_disabled       37352 non-null  bool 
 13  ratings_disabled        37352 non-null  bool 
 14  video_error_or_removed  37352 non-null  bool 
 15  description             36791 

### 4. Missing Values Audit

Why this check matters: Missing values in core metrics (views, video_id, category_id) can break downstream database integrity or skew BI calculations.

In [5]:
missing_series = df.isnull().sum()
missing_pct = (missing_series / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing_series,
    'Missing_Percentage (%)': missing_pct.round(2)
})

missing_df[missing_df['Missing_Count'] > 0]

,Missing_Count,Missing_Percentage (%)
description,561,1.50


> **Finding:** Only the `description` column has missing values (561 rows, 1.50%). All other 15 columns have 100% completeness.

### 5. Duplicate Rows vs. Longitudinal Daily Snapshots

Why this check matters: In media analytics, a video trending for 5 consecutive days is captured 5 times. These are **not duplicates**; they represent trending persistence. However, exact identical rows on the same day represent redundant web scraping errors.

In [6]:
exact_duplicates = df.duplicated().sum()
same_day_video_dups = df.duplicated(subset=['video_id', 'trending_date']).sum()

print(f"Exact Duplicate Rows (all columns identical): {exact_duplicates:,} ({exact_duplicates/len(df)*100:.2f}%)")
print(f"Same-Day Snapshots for same video_id: {same_day_video_dups:,} ({same_day_video_dups/len(df)*100:.2f}%)")

# Persistence Analysis: How many days does a video trend?
trend_frequency = df['video_id'].value_counts()
print(f"Unique Videos: {df['video_id'].nunique():,}")
print(f"Average trending days per video: {trend_frequency.mean():.2f}")
print("Top 5 most frequent video IDs in dataset:")
print(trend_frequency.head(5))

Exact Duplicate Rows (all columns identical): 4,263 (11.41%)
Same-Day Snapshots for same video_id: 4,894 (13.10%)
Unique Videos: 16,307
Average trending days per video: 2.29
Top 5 most frequent video IDs in dataset:
video_id
#NAME?         511
rRr1qiJRsXk     16
1J76wN0TPI4     14
l7E0kTvARsA     14
yFFL1we4j_Y     13
Name: count, dtype: int64


### 6. Investigating the Microsoft Excel `#NAME?` Identifier Bug

Notice that `#NAME?` appears 511 times above. Why does this happen?
When a YouTube 11-character video ID begins with a leading hyphen (`-`) or equals sign (`=`), opening or saving the CSV in Microsoft Excel causes Excel to interpret the value as a formula error. This replaces the video ID with `#NAME?`.

In [7]:
name_err_df = df[df['video_id'] == '#NAME?']
print(f"Total corrupted '#NAME?' rows: {len(name_err_df):,}")
print(f"Distinct titles within '#NAME?' records: {name_err_df['title'].nunique():,}")
print(f"Distinct channels within '#NAME?' records: {name_err_df['channel_title'].nunique():,}")

# Sample records with #NAME?
name_err_df[['title', 'channel_title', 'publish_time', 'views']].head(3)

Total corrupted '#NAME?' rows: 511
Distinct titles within '#NAME?' records: 214
Distinct channels within '#NAME?' records: 168


,title,channel_title,publish_time,views
134,స‌మంత కంట‌త‌డి | Samantha became EMOTIONAL @ S...,Friday Poster,2017-11-13T08:59:27.000Z,31052
173,कुंभ राशि वालों के लिए 12 नवंबर - 18 नवंबर का ...,Jansatta,2017-11-11T09:09:06.000Z,30659
189,"घर में चुपचाप यहाँ रख दे एक लौंग , इतना बरसेगा...",Health Tips for You,2017-11-08T12:27:17.000Z,743321


> **Action for Phase 3 Cleaning:** We will generate deterministic surrogate keys (`SURR_{hash}`) for `#NAME?` records using MD5 hashing of `(channel_title, title, publish_time)` so distinct videos are not collapsed into a single entity.

### 7. Temporal Format and Date Range Analysis

The `trending_date` column is formatted as `YY.DD.MM`. Let us verify parsing with `pd.to_datetime(format='%y.%d.%m')`.

In [8]:
parsed_trending = pd.to_datetime(df['trending_date'], format='%y.%d.%m')
parsed_publish = pd.to_datetime(df['publish_time'])

print(f"Trending Date Range: {parsed_trending.min().strftime('%Y-%m-%d')} to {parsed_trending.max().strftime('%Y-%m-%d')}")
print(f"Total Trending Days: {(parsed_trending.max() - parsed_trending.min()).days} days")
print(f"Publish Time Range:  {parsed_publish.min().strftime('%Y-%m-%d')} to {parsed_publish.max().strftime('%Y-%m-%d')}")

Trending Date Range: 2017-11-14 to 2018-06-14
Total Trending Days: 212 days
Publish Time Range:  2017-05-27 to 2018-06-13


### 8. Numerical Metrics Sanity Check

We check views, likes, dislikes, and comments for negative numbers or extreme values.

In [9]:
metric_cols = ['views', 'likes', 'dislikes', 'comment_count']
df[metric_cols].describe()

,views,likes,dislikes,comment_count
count,37352.00,37352.00,37352.00,37352.00
mean,1060477.65,27082.72,1665.08,2677.00
std,3184932.05,97145.10,16076.17,14868.32
min,4024.00,0.00,0.00,0.00
25%,123915.50,864.00,108.00,81.00
50%,304586.00,3069.00,326.00,329.00
75%,799291.25,13774.25,1019.25,1285.00
max,125432237.00,2912710.00,1545017.00,827755.00


### 9. Category Mapping & The Category 29 Anomaly

Let us check whether all `category_id` values in the CSV map to known titles in `IN_category_id.json`.

In [10]:
csv_cats = set(df['category_id'].unique())
json_cats = set(category_map.keys())
unmapped = csv_cats - json_cats

print(f"Categories in CSV: {len(csv_cats)}")
print(f"Categories in JSON: {len(json_cats)}")
print(f"Unmapped Category IDs in CSV: {unmapped}")

if 29 in unmapped:
    count_29 = (df['category_id'] == 29).sum()
    print(f"Category ID 29 occurs {count_29} times in the CSV.")
    print("Standard YouTube Taxonomy: ID 29 = 'Nonprofits & Activism'")
    # Enrich category map
    category_map[29] = 'Nonprofits & Activism'

# Map categories and view distribution
df['category_title'] = df['category_id'].map(category_map)
print("\nTop Categories by Observation Count:")
print(df['category_title'].value_counts())

Categories in CSV: 17
Categories in JSON: 31
Unmapped Category IDs in CSV: {np.int64(29)}
Category ID 29 occurs 105 times in the CSV.
Standard YouTube Taxonomy: ID 29 = 'Nonprofits & Activism'

Top Categories by Observation Count:
category_title
Entertainment            16712
News & Politics           5241
Music                     3858
Comedy                    3429
People & Blogs            2624
Film & Animation          1658
Education                 1227
Howto & Style              845
Sports                     731
Science & Technology       552
Shows                      205
Nonprofits & Activism      105
Autos & Vehicles            72
Gaming                      66
Movies                      16
Travel & Events              8
Pets & Animals               3
Name: count, dtype: int64


### 10. Summary of Profiling Findings for Phase 3 Cleaning

1. **Deduplication:** Remove 4,263 exact duplicate rows.
2. **Composite Uniqueness:** Deduplicate same-day snapshots on `(video_id, trending_date)` keeping the latest record.
3. **Identifier Repair:** Generate surrogate keys for `#NAME?` records.
4. **Date Standardization:** Convert `trending_date` from `%y.%d.%m` to standard ISO `YYYY-MM-DD` and parse `publish_time`.
5. **Taxonomy Enrichment:** Resolve Category 29 as 'Nonprofits & Activism'.
6. **Missing Values:** Fill missing `description` with `""`.